# ASL Sign Recognition - Landmark Attention Transformer

A cleaner approach focusing on:
1. **Landmark Attention**: Learn which landmarks matter most
2. **Deeper Transformer**: More layers, more capacity
3. **Better Features**: Position + Velocity + Acceleration + Distances
4. **Robust Augmentation**: Proven techniques that work
5. **Ensemble-Ready**: Can be ensembled with other models

In [ ]:
import json
import os
import math
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader, Sampler

from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}, AMP: {use_amp}")

In [ ]:
# Configuration
BASE_PATH = "/kaggle/input/asl-signs"
TRAIN_FILE = "/kaggle/input/asl-signs/train.csv"
SIGN_INDEX_FILE = "/kaggle/input/asl-signs/sign_to_prediction_index_map.json"

with open(SIGN_INDEX_FILE, "r") as f:
    SIGN2INDEX = json.load(f)

NUM_CLASSES = len(SIGN2INDEX)
print(f"Classes: {NUM_CLASSES}")

In [ ]:
# Selected face landmarks (most informative for ASL)
FACE_LANDMARKS = {
    'nose': [1, 2, 4, 5, 6, 19, 94, 168, 197, 195],
    'left_eye': [33, 133, 160, 159, 158, 157, 173, 144, 145, 153, 154, 155, 156, 246, 7, 163],
    'right_eye': [263, 362, 387, 386, 385, 384, 398, 373, 374, 380, 381, 382, 466, 388, 390, 249],
    'left_eyebrow': [70, 63, 105, 66, 107, 55, 65, 52],
    'right_eyebrow': [300, 293, 334, 296, 336, 285, 295, 282],
    'mouth_outer': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
    'mouth_inner': [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95],
    'face_oval': [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 
                  152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
}
SELECTED_FACE = sorted(set(i for v in FACE_LANDMARKS.values() for i in v))
FACE_IDX_MAP = {orig: new for new, orig in enumerate(SELECTED_FACE)}

# Landmark structure
N_POSE = 33
N_HAND = 21
N_FACE = len(SELECTED_FACE)

print(f"Pose: {N_POSE}, Hand: {N_HAND}x2, Face: {N_FACE}")
print(f"Total landmarks: {N_POSE + 2*N_HAND + N_FACE}")

In [ ]:
def generate_columns():
    """Generate column names for the flattened feature vector"""
    axes = ['x', 'y']
    cols = []
    
    # Order: left_hand, pose, right_hand, face
    for i in range(N_HAND):
        for ax in axes:
            cols.append(f'left_hand_{i}_{ax}')
    
    for i in range(N_POSE):
        for ax in axes:
            cols.append(f'pose_{i}_{ax}')
    
    for i in range(N_HAND):
        for ax in axes:
            cols.append(f'right_hand_{i}_{ax}')
    
    for face_idx in SELECTED_FACE:
        for ax in axes:
            cols.append(f'face_{face_idx}_{ax}')
    
    return cols

ALL_COLUMNS = generate_columns()
N_LANDMARKS = N_POSE + 2 * N_HAND + N_FACE
COORD_DIM = len(ALL_COLUMNS)  # N_LANDMARKS * 2

print(f"Total coordinate features: {COORD_DIM}")

In [ ]:
def load_and_preprocess(file_path: str) -> np.ndarray:
    """Load parquet and return normalized coordinates"""
    df = pd.read_parquet(os.path.join(BASE_PATH, file_path))
    
    # Filter face landmarks
    face_df = df[df['type'] == 'face']
    face_df = face_df[face_df['landmark_index'].isin(SELECTED_FACE)]
    other_df = df[df['type'] != 'face']
    df = pd.concat([face_df, other_df], ignore_index=True)
    
    # Create unique ID for pivoting
    df['UID'] = df['type'] + '_' + df['landmark_index'].astype(str)
    df = df.sort_values(['frame', 'UID'])
    
    # Get nose position for normalization
    nose = df[(df['type'] == 'pose') & (df['landmark_index'] == 0)].set_index('frame')[['x', 'y']]
    
    # Normalize by nose
    def normalize(frame_df):
        frame = frame_df.name
        if frame in nose.index:
            frame_df[['x', 'y']] = frame_df[['x', 'y']].values - nose.loc[frame].values
        return frame_df
    
    df = df.groupby('frame', group_keys=False).apply(normalize)
    
    # Pivot to wide format
    pivot = df.pivot_table(index='frame', columns='UID', values=['x', 'y'])
    pivot.columns = [f'{col[1]}_{col[0]}' for col in pivot.columns]
    pivot = pivot.reindex(columns=ALL_COLUMNS)
    
    # Fill missing values
    data = pivot.ffill().bfill().fillna(0).values.astype(np.float32)
    
    return data

In [ ]:
def compute_features(coords: np.ndarray) -> np.ndarray:
    """
    Compute rich features from coordinates.
    Input: (T, N_LANDMARKS * 2)
    Output: (T, feature_dim)
    
    Features:
    - Position (original coordinates)
    - Velocity (1st derivative)
    - Acceleration (2nd derivative)
    - Key distances (hand shapes, arm positions)
    """
    T = coords.shape[0]
    
    # Velocity
    vel = np.zeros_like(coords)
    vel[1:] = coords[1:] - coords[:-1]
    
    # Acceleration
    acc = np.zeros_like(coords)
    acc[1:] = vel[1:] - vel[:-1]
    
    # Reshape to (T, N_LANDMARKS, 2) for distance computation
    pos_reshaped = coords.reshape(T, -1, 2)
    
    # Key distances (using landmark indices)
    # Left hand: 0-20, Pose: 21-53, Right hand: 54-74, Face: 75+
    
    # Hand landmark offsets in our ordering
    LH_OFFSET = 0
    POSE_OFFSET = N_HAND
    RH_OFFSET = N_HAND + N_POSE
    
    distances = []
    
    # Left hand distances
    lh_thumb_index = np.linalg.norm(pos_reshaped[:, LH_OFFSET + 4] - pos_reshaped[:, LH_OFFSET + 8], axis=1)
    lh_index_pinky = np.linalg.norm(pos_reshaped[:, LH_OFFSET + 8] - pos_reshaped[:, LH_OFFSET + 20], axis=1)
    lh_palm = np.linalg.norm(pos_reshaped[:, LH_OFFSET + 0] - pos_reshaped[:, LH_OFFSET + 9], axis=1)
    
    # Right hand distances  
    rh_thumb_index = np.linalg.norm(pos_reshaped[:, RH_OFFSET + 4] - pos_reshaped[:, RH_OFFSET + 8], axis=1)
    rh_index_pinky = np.linalg.norm(pos_reshaped[:, RH_OFFSET + 8] - pos_reshaped[:, RH_OFFSET + 20], axis=1)
    rh_palm = np.linalg.norm(pos_reshaped[:, RH_OFFSET + 0] - pos_reshaped[:, RH_OFFSET + 9], axis=1)
    
    # Pose distances (shoulder, wrist positions)
    # Pose indices: 11=left_shoulder, 12=right_shoulder, 15=left_wrist, 16=right_wrist
    shoulder_width = np.linalg.norm(pos_reshaped[:, POSE_OFFSET + 11] - pos_reshaped[:, POSE_OFFSET + 12], axis=1)
    wrist_distance = np.linalg.norm(pos_reshaped[:, POSE_OFFSET + 15] - pos_reshaped[:, POSE_OFFSET + 16], axis=1)
    left_arm_ext = np.linalg.norm(pos_reshaped[:, POSE_OFFSET + 11] - pos_reshaped[:, POSE_OFFSET + 15], axis=1)
    right_arm_ext = np.linalg.norm(pos_reshaped[:, POSE_OFFSET + 12] - pos_reshaped[:, POSE_OFFSET + 16], axis=1)
    
    # Hand to face distances (important for many signs)
    # Nose is at pose index 0
    left_hand_to_nose = np.linalg.norm(pos_reshaped[:, LH_OFFSET + 9] - pos_reshaped[:, POSE_OFFSET + 0], axis=1)
    right_hand_to_nose = np.linalg.norm(pos_reshaped[:, RH_OFFSET + 9] - pos_reshaped[:, POSE_OFFSET + 0], axis=1)
    
    distances = np.stack([
        lh_thumb_index, lh_index_pinky, lh_palm,
        rh_thumb_index, rh_index_pinky, rh_palm,
        shoulder_width, wrist_distance, left_arm_ext, right_arm_ext,
        left_hand_to_nose, right_hand_to_nose
    ], axis=1)  # (T, 12)
    
    # Distance velocity
    dist_vel = np.zeros_like(distances)
    dist_vel[1:] = distances[1:] - distances[:-1]
    
    # Concatenate all features
    features = np.concatenate([
        coords,      # Position
        vel,         # Velocity
        acc,         # Acceleration
        distances,   # Key distances
        dist_vel     # Distance velocity
    ], axis=1)
    
    return features.astype(np.float32)

In [ ]:
class ASLDataset(Dataset):
    def __init__(self, features_list, labels, max_frames=128, augment=False):
        self.features = features_list
        self.labels = labels
        self.max_frames = max_frames
        self.augment = augment
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        x = self.features[idx].copy()
        y = self.labels[idx]
        
        if self.augment:
            x = self._augment(x)
        
        # Subsample if too long
        if x.shape[0] > self.max_frames:
            indices = np.linspace(0, x.shape[0] - 1, self.max_frames).astype(int)
            x = x[indices]
        
        return torch.tensor(x, dtype=torch.float32), y
    
    def _augment(self, x):
        T = x.shape[0]
        
        # Temporal scaling (speed variation)
        if np.random.random() > 0.5:
            scale = np.random.uniform(0.8, 1.2)
            new_T = max(5, int(T * scale))
            indices = np.linspace(0, T - 1, new_T).astype(int)
            x = x[indices]
            T = new_T
        
        # Gaussian noise
        if np.random.random() > 0.5:
            noise = np.random.normal(0, 0.003, x.shape)
            x = x + noise
        
        # Temporal crop
        if np.random.random() > 0.5 and T > 20:
            start = np.random.randint(0, max(1, T // 10))
            end = T - np.random.randint(0, max(1, T // 10))
            if end > start + 5:
                x = x[start:end]
        
        # Spatial shift
        if np.random.random() > 0.5:
            shift = np.random.uniform(-0.02, 0.02, (1, x.shape[1]))
            x = x + shift
        
        # Frame dropout (randomly zero out some frames)
        if np.random.random() > 0.7:
            drop_ratio = np.random.uniform(0.05, 0.15)
            drop_mask = np.random.random(x.shape[0]) > drop_ratio
            x = x * drop_mask[:, np.newaxis]
        
        return x.astype(np.float32)

In [ ]:
def collate_fn(batch):
    sequences, labels = zip(*batch)
    
    lengths = [s.shape[0] for s in sequences]
    max_len = max(lengths)
    B = len(sequences)
    D = sequences[0].shape[1]
    
    padded = torch.zeros(B, max_len, D)
    mask = torch.zeros(B, max_len, dtype=torch.bool)
    
    for i, seq in enumerate(sequences):
        T = seq.shape[0]
        padded[i, :T] = seq
        mask[i, :T] = True
    
    return padded, mask, torch.tensor(labels)


class BucketSampler(Sampler):
    def __init__(self, lengths, batch_size):
        self.lengths = lengths
        self.batch_size = batch_size
    
    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = [indices[i:i+self.batch_size] for i in range(0, len(indices), self.batch_size)]
        np.random.shuffle(batches)
        for batch in batches:
            yield list(batch)
    
    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

In [ ]:
# ============== MODEL ==============

class LandmarkAttention(nn.Module):
    """Learn attention weights for different landmarks"""
    def __init__(self, n_landmarks, hidden_dim=64):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(n_landmarks * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_landmarks),
            nn.Softmax(dim=-1)
        )
        self.n_landmarks = n_landmarks
    
    def forward(self, x):
        # x: (B, T, N_landmarks * 2) - position features only
        B, T, _ = x.shape
        
        # Compute attention per frame
        attn = self.attention(x)  # (B, T, N_landmarks)
        
        # Reshape x to (B, T, N_landmarks, 2)
        x_reshaped = x.view(B, T, self.n_landmarks, 2)
        
        # Apply attention
        attn = attn.unsqueeze(-1)  # (B, T, N_landmarks, 1)
        weighted = x_reshaped * attn  # (B, T, N_landmarks, 2)
        
        return weighted.view(B, T, -1), attn.squeeze(-1)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(1)])

In [ ]:
class ASLTransformer(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, n_heads=8, 
                 n_layers=6, dropout=0.15, n_landmarks=209):
        super().__init__()
        
        # Landmark attention (applied to position features only)
        self.landmark_attn = LandmarkAttention(n_landmarks, hidden_dim=64)
        
        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        
        # Output layers
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
        
        self.n_landmarks = n_landmarks
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x, mask):
        B, T, D = x.shape
        
        # Apply landmark attention to position features (first N_LANDMARKS * 2 features)
        pos_dim = self.n_landmarks * 2
        pos_features = x[:, :, :pos_dim]
        other_features = x[:, :, pos_dim:]
        
        weighted_pos, attn_weights = self.landmark_attn(pos_features)
        x = torch.cat([weighted_pos, other_features], dim=-1)
        
        # Project to model dimension
        x = self.input_proj(x)
        x = self.pos_enc(x)
        
        # Add CLS token
        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        
        # Update mask
        cls_mask = torch.ones(B, 1, device=mask.device, dtype=torch.bool)
        mask = torch.cat([cls_mask, mask], dim=1)
        
        # Transformer
        x = self.encoder(x, src_key_padding_mask=~mask)
        x = self.norm(x[:, 0])  # CLS token
        
        return self.classifier(x)

In [ ]:
# ============== LOSS FUNCTIONS ==============

class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        n_classes = pred.size(-1)
        
        # Create smoothed labels
        with torch.no_grad():
            smooth_target = torch.zeros_like(pred)
            smooth_target.fill_(self.smoothing / (n_classes - 1))
            smooth_target.scatter_(1, target.unsqueeze(1), 1 - self.smoothing)
        
        log_probs = F.log_softmax(pred, dim=-1)
        loss = (-smooth_target * log_probs).sum(dim=-1).mean()
        
        return loss

In [ ]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ============== DATA LOADING ==============

def load_all_data():
    train_df = pd.read_csv(TRAIN_FILE)
    train_df['sign'] = train_df['sign'].map(SIGN2INDEX)
    
    train_split, val_split = train_test_split(
        train_df, test_size=0.1, stratify=train_df['sign'], random_state=42
    )
    
    print(f"Loading {len(train_split)} training samples...")
    train_features = []
    for path in tqdm(train_split['path'].tolist()):
        try:
            coords = load_and_preprocess(path)
            features = compute_features(coords)
            train_features.append(features)
        except Exception as e:
            print(f"Error: {path} - {e}")
            # Dummy features
            train_features.append(np.zeros((10, COORD_DIM * 3 + 24), dtype=np.float32))
    
    print(f"Loading {len(val_split)} validation samples...")
    val_features = []
    for path in tqdm(val_split['path'].tolist()):
        try:
            coords = load_and_preprocess(path)
            features = compute_features(coords)
            val_features.append(features)
        except Exception as e:
            print(f"Error: {path} - {e}")
            val_features.append(np.zeros((10, COORD_DIM * 3 + 24), dtype=np.float32))
    
    return (
        train_features, train_split['sign'].to_numpy(),
        val_features, val_split['sign'].to_numpy()
    )

In [ ]:
# Load data
train_features, train_labels, val_features, val_labels = load_all_data()

# Get feature dimension
FEATURE_DIM = train_features[0].shape[1]
print(f"Feature dimension: {FEATURE_DIM}")

In [ ]:
# Create datasets and loaders
MAX_FRAMES = 128
BATCH_SIZE = 64

train_dataset = ASLDataset(train_features, train_labels, MAX_FRAMES, augment=True)
val_dataset = ASLDataset(val_features, val_labels, MAX_FRAMES, augment=False)

train_lengths = [min(f.shape[0], MAX_FRAMES) for f in train_features]
val_lengths = [min(f.shape[0], MAX_FRAMES) for f in val_features]

train_loader = DataLoader(
    train_dataset, 
    batch_sampler=BucketSampler(train_lengths, BATCH_SIZE),
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_sampler=BucketSampler(val_lengths, BATCH_SIZE),
    collate_fn=collate_fn
)

print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

In [ ]:
# ============== MODEL SETUP ==============

model = ASLTransformer(
    input_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES,
    d_model=256,
    n_heads=8,
    n_layers=6,
    dropout=0.15,
    n_landmarks=N_LANDMARKS
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {num_params:,}")

In [ ]:
# Training setup
EPOCHS = 100
WARMUP_EPOCHS = 5
PATIENCE = 20
MIXUP_ALPHA = 0.2

criterion = LabelSmoothingCE(smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

# OneCycleLR for better convergence
scheduler = OneCycleLR(
    optimizer,
    max_lr=3e-4,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy='cos'
)

scaler = GradScaler(enabled=use_amp)

In [ ]:
def train_epoch(loader, use_mixup=True):
    model.train()
    total_loss = 0
    
    for x, mask, y in loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=use_amp, device_type=device.type):
            if use_mixup and np.random.random() > 0.5:
                x, y_a, y_b, lam = mixup_data(x, y, MIXUP_ALPHA)
                logits = model(x, mask)
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                logits = model(x, mask)
                loss = criterion(logits, y)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, mask, y in loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)
        
        with autocast(enabled=use_amp, device_type=device.type):
            logits = model(x, mask)
            loss = criterion(logits, y)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
# ============== TRAINING ==============

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader, use_mixup=True)
    val_loss, val_acc = validate_epoch(val_loader)
    
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f} | LR: {lr:.2e}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
        }, 'best_model.pth')
        print(f"  -> Best model saved! Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest validation accuracy: {best_acc:.4f}")

In [ ]:
# Load best and evaluate
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss, val_acc = validate_epoch(val_loader)
print(f"Final validation accuracy: {val_acc:.4f}")